In [1]:
from assignment_2_2024.msg import PlanningAction, PlanningGoal

import time
import random
import pickle
import rospy
import actionlib
import threading

In [ ]:
from gazebo_msgs.msg import ModelState
from gazebo_msgs.srv import SetModelState

x_init = 1
y_init = 1


def teleport_robot():
    try:     
        rospy.wait_for_service('/gazebo/set_model_state')
        set_state = rospy.ServiceProxy('/gazebo/set_model_state', SetModelState)
        state_msg = ModelState()
        state_msg.pose.position.x = x_init
        state_msg.pose.position.y = y_init
        state_msg.pose.position.z = 0
        state_msg.pose.orientation.x = 0
        state_msg.pose.orientation.y = 0
        state_msg.pose.orientation.z = 0
        state_msg.pose.orientation.w = 1
        state_msg.reference_frame = 'world'
        state_msg.model_name = 'robot1'

        set_state(state_msg)
        rospy.loginfo("Robot teleported to initial position.")
    except:
        pass


def time_goals(goals):
    times = []

    for (x, y) in goals:
        start = time.time()
        teleport_robot()
        
        old_reached = reached_goals

        update_goal(x, y)
        rospy.loginfo(f"Setting the goal at ({x}, {y})")

        while time.time() - start < 300.0:
            if reached_goals > old_reached:
                times.append(time.time() - start)
                break
        else:
            times.append(0.0)
            rospy.loginfo(f"Goal ({x}, {y}) not reached in the max time window!")
            
    return times

In [3]:
active_goal = False
not_reached_goals = 0

def update_goal(x, y):
    global active_goal, not_reached_goals
    goal = PlanningGoal()
    
    if active_goal:
        not_reached_goals = not_reached_goals + 1
    else:
        active_goal = True

    goal.target_pose.header.frame_id = "map"
    goal.target_pose.pose.position.x = x
    goal.target_pose.pose.position.y = y
    
    client.send_goal(goal, feedback_cb=read_feedback)

In [4]:
active_goal = False
latest_feedback = None
reached_goals = 0

def read_feedback(feedback):
    global active_goal, latest_feedback, reached_goals
    latest_feedback = feedback

    if (active_goal and feedback.stat == 'Target reached!'):
        reached_goals = reached_goals + 1
        active_goal = False
        goal = None

In [5]:
rospy.init_node('action_client')
client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
client.wait_for_server()

True

In [6]:
random.seed(10)

goals = [ (random.randrange(-9, 9, 1), random.randrange(-9, 9, 1)) for i in range(100) ]

with open("goals.pkl", "wb") as f:
    pickle.dump(goals, f)

times = time_goals(goals)

with open("times.pkl", "wb") as f:
    pickle.dump(times, f)

[INFO] [1748263182.568487, 1220.838000]: Robot teleported to initial position.
[INFO] [1748263182.576004, 1220.846000]: Setting the goal at (-8, 4)
[INFO] [1748263216.243311, 1250.964000]: Robot teleported to initial position.
[INFO] [1748263216.248953, 1250.968000]: Setting the goal at (6, -9)
[INFO] [1748263314.243257, 1337.617000]: Robot teleported to initial position.
[INFO] [1748263314.254071, 1337.623000]: Setting the goal at (-3, 5)
[INFO] [1748263422.539436, 1429.152000]: Robot teleported to initial position.
[INFO] [1748263422.546512, 1429.152000]: Setting the goal at (6, -1)
[INFO] [1748263722.511222, 1683.921000]: Goal (6, -1) not reached in the max time window!
[INFO] [1748263722.576731, 1683.992000]: Robot teleported to initial position.
[INFO] [1748263722.581456, 1683.994000]: Setting the goal at (-4, -8)
[INFO] [1748263796.075145, 1746.835000]: Robot teleported to initial position.
[INFO] [1748263796.088758, 1746.847000]: Setting the goal at (7, 6)
[INFO] [1748264096.034

[INFO] [1748269562.109266, 6680.676000]: Robot teleported to initial position.
[INFO] [1748269562.116534, 6680.681000]: Setting the goal at (1, 1)
[INFO] [1748269562.276215, 6680.810000]: Robot teleported to initial position.
[INFO] [1748269562.282615, 6680.816000]: Setting the goal at (-2, -7)
[INFO] [1748269637.275285, 6747.728000]: Robot teleported to initial position.
[INFO] [1748269637.281428, 6747.733000]: Setting the goal at (-1, 5)
[INFO] [1748269937.256677, 7011.575000]: Goal (-1, 5) not reached in the max time window!
[INFO] [1748269937.289676, 7011.602000]: Robot teleported to initial position.
[INFO] [1748269937.295868, 7011.611000]: Setting the goal at (3, -4)
[INFO] [1748270028.430807, 7092.653000]: Robot teleported to initial position.
[INFO] [1748270028.439186, 7092.661000]: Setting the goal at (3, 6)
[INFO] [1748270081.030715, 7138.874000]: Robot teleported to initial position.
[INFO] [1748270081.037256, 7138.880000]: Setting the goal at (-2, 7)
[INFO] [1748270173.7489

[INFO] [1748274851.294404, 11351.681000]: Robot teleported to initial position.
[INFO] [1748274851.300959, 11351.686000]: Setting the goal at (4, 3)
[INFO] [1748274866.008457, 11364.459000]: Robot teleported to initial position.
[INFO] [1748274866.014742, 11364.464000]: Setting the goal at (-3, -7)
[INFO] [1748274942.537008, 11431.385000]: Robot teleported to initial position.
[INFO] [1748274942.565515, 11431.391000]: Setting the goal at (4, -7)


In [7]:
def spin():
    rospy.spin()

spin_thread = threading.Thread(target=spin)
spin_thread.start()